In [58]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Input directories and subjects（既存のセルから再利用）
base_pred_dir = "/home/tsukagoshitoshihiro/workspace/APSIPA2025/Semi_supervise/test_output/1st_pseudo_label_source"
base_gt_dir = "/home/tsukagoshitoshihiro/workspace/APSIPA2025/Semi_supervise/dataset/exist_label"

# 評価する被験者リスト
subjects = ["301", "307", "309"]  # 評価したい被験者IDをここに追加

# ファイルパス生成関数（既存のセルから再利用）
def get_file_paths(subject_id):
    pred_file = f"{base_pred_dir}/WavLM-base+GRU_{subject_id}_frame_probabilities.txt"
    gt_file = f"{base_gt_dir}/{subject_id}.txt"
    return pred_file, gt_file

# Evaluation parameters
labels_to_evaluate = {
    "chewing": "ch",
    "swallowing": "sw"
}

# Time window for evaluation (in seconds)
EVAL_START_SEC = 10 * 60  # 10 minutes
EVAL_END_SEC = 30 * 60    # 30 minutes

# 使用する閾値
threshold = 0.5  # 予測確率の閾値

# IoUの評価範囲 - 重複がないように配列を作成
iou_ranges = np.round(np.arange(0.0, 1.01, 0.1), 2)  # 0.0から1.0までの範囲を明示的に指定

# --- Utility Functions ---

def convert_to_segments(binary_series: np.ndarray, times: np.ndarray) -> list:
    segments = []
    in_segment = False
    start_time = 0.0
    padded_series = np.pad(binary_series, (0, 1), 'constant', constant_values=0)
    padded_times = np.pad(times, (0, 1), 'edge')

    for i, value in enumerate(padded_series):
        if value > 0 and not in_segment:
            in_segment = True
            start_time = padded_times[i]
        elif value == 0 and in_segment:
            in_segment = False
            end_time = padded_times[i]
            if start_time < end_time:
                segments.append((start_time, end_time))
    return segments

def calculate_iou(seg1: tuple, seg2: tuple) -> float:
    start1, end1 = seg1
    start2, end2 = seg2
    intersection_start = max(start1, start2)
    intersection_end = min(end1, end2)
    intersection = max(0, intersection_end - intersection_start)
    union = (end1 - start1) + (end2 - start2) - intersection
    return intersection / union if union > 0 else 0

def evaluate_performance(gt_segments: list, pred_segments: list, iou_threshold: float) -> tuple:
    if not gt_segments or not pred_segments:
        tp, fp, fn = 0, len(pred_segments), len(gt_segments)
    else:
        iou_matrix = np.zeros((len(gt_segments), len(pred_segments)))
        for i, gt_seg in enumerate(gt_segments):
            for j, pred_seg in enumerate(pred_segments):
                iou_matrix[i, j] = calculate_iou(gt_seg, pred_seg)

        matched_gt = np.zeros(len(gt_segments), dtype=bool)
        matched_pred = np.zeros(len(pred_segments), dtype=bool)

        epsilon = 1e-6
        potential_matches = np.where(iou_matrix >= max(iou_threshold, epsilon))

        sorted_indices = np.argsort(iou_matrix[potential_matches])[::-1]

        for idx in sorted_indices:
            gt_idx, pred_idx = potential_matches[0][idx], potential_matches[1][idx]
            if not matched_gt[gt_idx] and not matched_pred[pred_idx]:
                iou = iou_matrix[gt_idx, pred_idx]
                #print(f"Matched GT {gt_segments[gt_idx]} with Pred {pred_segments[pred_idx]}: IoU = {iou:.3f}")
                matched_gt[gt_idx], matched_pred[pred_idx] = True, True

        tp = np.sum(matched_gt)
        fn = len(gt_segments) - tp
        fp = len(pred_segments) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1_score, tp, fp, fn


# 1人の被験者のデータを処理する関数
def process_subject_iou_evaluation(subject_id, threshold):
    prediction_prob_file, ground_truth_file = get_file_paths(subject_id)
    
    try:
        pred_prob_data = pd.read_csv(prediction_prob_file, sep='\t')
        gt_data = pd.read_csv(ground_truth_file, sep='\t', header=None, names=['start_time', 'end_time', 'label'])
    except FileNotFoundError as e:
        print(f"Error for subject {subject_id}: Could not find file - {e.filename}")
        return None
    
    # 正解データをセグメントに変換
    ground_truth_events = {}
    for label_long, label_short in labels_to_evaluate.items():
        gt_segs_df = gt_data[gt_data['label'] == label_short]
        all_segments = list(zip(gt_segs_df['start_time'], gt_segs_df['end_time']))
        ground_truth_events[label_long] = [
            seg for seg in all_segments if seg[0] < EVAL_END_SEC and seg[1] > EVAL_START_SEC
        ]
    
    # 閾値での予測を生成
    predicted_events = {}
    for label_long in labels_to_evaluate.keys():
        binary_series = (pred_prob_data[label_long] >= threshold).astype(int).values
        all_segments = convert_to_segments(binary_series, pred_prob_data['time'].values)
        predicted_events[label_long] = [
            seg for seg in all_segments if seg[0] < EVAL_END_SEC and seg[1] > EVAL_START_SEC
        ]
    
    # 各IoU閾値ごとの評価結果を格納
    subject_results = {label: {iou: {'precision': 0, 'recall': 0, 'f1': 0} for iou in iou_ranges} 
                      for label in labels_to_evaluate.keys()}
    
    # 各ラベルについて処理
    for label_long in labels_to_evaluate.keys():
        gt_segs = ground_truth_events[label_long]
        pred_segs = predicted_events[label_long]
        
        # 各IoU閾値ごとに評価
        for iou_thresh in iou_ranges:
            precision, recall, f1_score, _, _, _ = evaluate_performance(gt_segs, pred_segs, iou_thresh)
            subject_results[label_long][iou_thresh]['precision'] = precision
            subject_results[label_long][iou_thresh]['recall'] = recall
            subject_results[label_long][iou_thresh]['f1'] = f1_score
    
    return subject_results

# --- 複数被験者の処理と平均値の計算 ---

print(f"処理する被験者: {', '.join(subjects)}")
print(f"予測確率の閾値: {threshold}")

# 全被験者の結果を保存
all_subject_results = {}

# 各被験者のデータを処理
for subject in subjects:
    print(f"\n処理中: 被験者 {subject}...")
    subject_results = process_subject_iou_evaluation(subject, threshold)
    if subject_results:
        all_subject_results[subject] = subject_results
        
# 平均値を計算
avg_results = {label: {iou: {'precision': [], 'recall': [], 'f1': []} for iou in iou_ranges} 
               for label in labels_to_evaluate.keys()}

# 各被験者の結果を集計
for subject, results in all_subject_results.items():
    for label in labels_to_evaluate.keys():
        for iou in iou_ranges:
            avg_results[label][iou]['precision'].append(results[label][iou]['precision'])
            avg_results[label][iou]['recall'].append(results[label][iou]['recall'])
            avg_results[label][iou]['f1'].append(results[label][iou]['f1'])

# 平均値の計算
for label in labels_to_evaluate.keys():
    for iou in iou_ranges:
        avg_results[label][iou]['precision'] = np.mean(avg_results[label][iou]['precision'])
        avg_results[label][iou]['recall'] = np.mean(avg_results[label][iou]['recall'])
        avg_results[label][iou]['f1'] = np.mean(avg_results[label][iou]['f1'])

# --- 各被験者の評価結果を表示 ---
print("\n===== 各被験者のIoU別評価結果 =====")
for subject, results in all_subject_results.items():
    print(f"\n被験者 {subject}:")
    for label in labels_to_evaluate.keys():
        print(f"  {label}:")
        for iou in iou_ranges:
            precision = results[label][iou]['precision']
            recall = results[label][iou]['recall']
            f1 = results[label][iou]['f1']
            print(f"    IoU > {iou:.1f}: P={precision:.3f}, R={recall:.3f}, F1={f1:.3f}")

# --- 平均評価結果を表示 ---
print("\n===== 平均IoU別評価結果 =====")
for label in labels_to_evaluate.keys():
    print(f"\n{label}:")
    for iou in iou_ranges:
        precision = avg_results[label][iou]['precision']
        recall = avg_results[label][iou]['recall']
        f1 = avg_results[label][iou]['f1']
        print(f"  IoU > {iou:.1f}: P={precision:.3f}, R={recall:.3f}, F1={f1:.3f}")
"""
# --- F1スコアのグラフを表示 ---
plt.figure(figsize=(12, 8))
for label in labels_to_evaluate.keys():
    f1_scores = [avg_results[label][iou]['f1'] for iou in iou_ranges]
    plt.plot(iou_ranges, f1_scores, 'o-', label=f'{label} - F1 Score')

plt.xlabel('IoU Threshold')
plt.ylabel('F1 Score')
plt.title(f'平均F1スコア vs IoU閾値 (予測確率閾値: {threshold})')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
"""

処理する被験者: 301, 307, 309
予測確率の閾値: 0.5

処理中: 被験者 301...

処理中: 被験者 307...

処理中: 被験者 307...

処理中: 被験者 309...

処理中: 被験者 309...

===== 各被験者のIoU別評価結果 =====

被験者 301:
  chewing:
    IoU > 0.0: P=0.888, R=0.672, F1=0.765
    IoU > 0.1: P=0.888, R=0.672, F1=0.765
    IoU > 0.2: P=0.845, R=0.640, F1=0.728
    IoU > 0.3: P=0.807, R=0.610, F1=0.695
    IoU > 0.4: P=0.734, R=0.555, F1=0.632
    IoU > 0.5: P=0.575, R=0.435, F1=0.495
    IoU > 0.6: P=0.361, R=0.273, F1=0.311
    IoU > 0.7: P=0.189, R=0.143, F1=0.163
    IoU > 0.8: P=0.086, R=0.065, F1=0.074
    IoU > 0.9: P=0.034, R=0.026, F1=0.030
    IoU > 1.0: P=0.000, R=0.000, F1=0.000
  swallowing:
    IoU > 0.0: P=0.887, R=0.855, F1=0.870
    IoU > 0.1: P=0.868, R=0.836, F1=0.852
    IoU > 0.2: P=0.868, R=0.836, F1=0.852
    IoU > 0.3: P=0.868, R=0.836, F1=0.852
    IoU > 0.4: P=0.868, R=0.836, F1=0.852
    IoU > 0.5: P=0.755, R=0.727, F1=0.741
    IoU > 0.6: P=0.698, R=0.673, F1=0.685
    IoU > 0.7: P=0.566, R=0.545, F1=0.556
    IoU > 0.8: P=0.

"\n# --- F1スコアのグラフを表示 ---\nplt.figure(figsize=(12, 8))\nfor label in labels_to_evaluate.keys():\n    f1_scores = [avg_results[label][iou]['f1'] for iou in iou_ranges]\n    plt.plot(iou_ranges, f1_scores, 'o-', label=f'{label} - F1 Score')\n\nplt.xlabel('IoU Threshold')\nplt.ylabel('F1 Score')\nplt.title(f'平均F1スコア vs IoU閾値 (予測確率閾値: {threshold})')\nplt.grid(True)\nplt.legend()\nplt.tight_layout()\nplt.show()\n"